# AlphaFold2 Structure Prediction: Defensin_beta (PF00711)

This notebook predicts structures for SA-generated and stored beta-defensin sequences
using AlphaFold2 via ColabFold. These are short peptides (~36 residues), so predictions
should complete quickly.

**Before running:**
1. Select GPU runtime (Runtime > Change runtime type > T4 GPU)
2. Upload `colabfold_input/PF00711/sa_generation.fasta` (50 sequences)
3. Upload `colabfold_input/PF00711/stored.fasta` (45 sequences)

Upload into `/content/colabfold_input/PF00711/` so the paths match.

In [ ]:
# Cell 1: Install ColabFold
import os
if not os.path.isfile("COLABFOLD_READY"):
    print("Installing ColabFold...")
    os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
    os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")
    os.system("touch COLABFOLD_READY")
    print("Done! If you see dependency warnings above, ignore them.")
else:
    print("ColabFold already installed.")

In [ ]:
# Cell 2: Mount Google Drive for automatic backups
from google.colab import drive
drive.mount('/content/drive')

BACKUP_DIR = "/content/drive/MyDrive/colabfold_backup"
import os
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"Backup directory: {BACKUP_DIR}")

In [ ]:
# Cell 3: Run AlphaFold2 predictions for Defensin_beta
import glob, os, shutil
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type

setup_logging(Path("log.txt"))
download_alphafold_params("alphafold2_ptm", Path(default_data_dir))

BACKUP_DIR = "/content/drive/MyDrive/colabfold_backup"
FAM_ID = "PF00711"  # Defensin_beta (~36 residues)
CATEGORIES = ["sa_generation", "stored"]

def backup_category(fam_id, cat):
    """Copy completed PDB results to Google Drive."""
    src = f"colabfold_output/{fam_id}/{cat}"
    dst = f"{BACKUP_DIR}/{fam_id}/{cat}"
    os.makedirs(dst, exist_ok=True)
    n_copied = 0
    for f in glob.glob(f"{src}/*.pdb"):
        shutil.copy2(f, dst)
        n_copied += 1
    print(f"  Backed up {n_copied} PDB files to Drive: {fam_id}/{cat}")

for cat in CATEGORIES:
    fasta = f"colabfold_input/{FAM_ID}/{cat}.fasta"
    outdir = f"colabfold_output/{FAM_ID}/{cat}"

    if not os.path.isfile(fasta):
        print(f"  {FAM_ID}/{cat}: FASTA not found at {fasta}, skipping")
        continue

    n_input = sum(1 for l in open(fasta) if l.startswith('>'))
    os.makedirs(outdir, exist_ok=True)
    n_done = len(glob.glob(f"{outdir}/*rank_001*.pdb"))

    if n_done >= n_input:
        print(f"  {FAM_ID}/{cat}: DONE ({n_done}/{n_input})")
        continue

    # Check if Drive backup has results we can restore
    backup_pdbs = glob.glob(f"{BACKUP_DIR}/{FAM_ID}/{cat}/*.pdb")
    if backup_pdbs and n_done == 0:
        print(f"  Restoring {len(backup_pdbs)} PDB files from Drive backup...")
        for f in backup_pdbs:
            shutil.copy2(f, outdir)
        n_done = len(glob.glob(f"{outdir}/*rank_001*.pdb"))
        if n_done >= n_input:
            print(f"  {FAM_ID}/{cat}: RESTORED ({n_done}/{n_input})")
            continue

    print(f"\n{'='*60}")
    print(f"  {FAM_ID}/{cat}: {n_input} sequences ({n_done} done)")
    print(f"{'='*60}")

    queries, is_complex = get_queries(fasta)
    run(
        queries=queries,
        result_dir=outdir,
        use_templates=False,
        num_relax=0,
        msa_mode="MMseqs2 (UniRef+Environmental)",
        model_type="alphafold2_ptm",
        num_models=1,
        num_recycles=3,
        model_order=[1],
        is_complex=is_complex,
        data_dir=Path(default_data_dir),
        keep_existing_results=True,
        rank_by="auto",
        stop_at_score=float(100),
        zip_results=False,
        user_agent="colabfold/google-colab-main",
    )

    backup_category(FAM_ID, cat)

print("\nDefensin_beta predictions complete!")

In [ ]:
# Cell 4: Check completion status
import glob, os

FAM_ID = "PF00711"
for cat in ["sa_generation", "stored"]:
    fasta = f"colabfold_input/{FAM_ID}/{cat}.fasta"
    outdir = f"colabfold_output/{FAM_ID}/{cat}"
    if os.path.isfile(fasta):
        n_in = sum(1 for l in open(fasta) if l.startswith('>'))
        n_out = len(glob.glob(f"{outdir}/*rank_001*.pdb"))
        status = 'DONE' if n_out >= n_in else f'{n_out}/{n_in}'
        print(f"  {FAM_ID}/{cat}: {status}")

In [ ]:
# Cell 5: Download results
import shutil
from google.colab import files

shutil.make_archive('colabfold_defensin_beta', 'zip', '.', 'colabfold_output')
files.download('colabfold_defensin_beta.zip')
print("Download complete. Unzip and copy PF00711/ into code/structure-validation/data/")